In [8]:
import os
from pathlib import Path 
from dataclasses import dataclass
import pandas as pd
import re

In [9]:
@dataclass
class DataCleaningConfig:
    root_dir : str
    source_path : str 
    local_data_file : str



In [10]:
os.chdir("c:\\Users\\RSR\\PYTHON\\LLMIssueTracker\src")

In [11]:
from LLMIssueTracker.Utils.Common import *
from LLMIssueTracker.Constant import *


In [ ]:
import os
from pathlib import Path

class ConfigurationManager:

    def __init__(
        self,
        Config_path=CONFIG_FILE_PATH,
        Params_Path=PARAMS_FILE_PATH
    ):

        print(Config_path)


        os.chdir(r"C:\Users\RSR\PYTHON\LLMIssueTracker")

        print(f"path config yaml {Path(Config_path)}")

        self.config = Read_Yaml(Path(Config_path))

        self.params = Read_Yaml(Path(Params_Path))
        print("CONFIG CONTENT:")
        print(self.config)

        print("AVAILABLE KEYS:")
        print(list(self.config.keys()))
        self.Data_Cleaning = self.config.Data_Cleaning
        
    def get_data_cleaning_config(self) -> DataCleaningConfig:

        config = self.Data_Cleaning

        create_directories([config.root_dir])

        data_cleaning_config = DataCleaningConfig(
            root_dir=config.root_dir,
            source_path=config.source_path,
            local_data_file=config.local_data_file
            
        )

        return data_cleaning_config

In [13]:
class DataCleaning:
    def __init__(self,config:DataCleaningConfig):
        self.config=config
        self.save_file = Path(self.config.local_data_file)
        

    def Data_Preprocessing(self)->pd.DataFrame:
        file=Path(self.config.source_path)
        print("Reading:", file)

        #file_path = Path(r"C:\Users\RSR\PYTHON\LLMIssueTracker\artifacts\DataIngestion\com_data_log.csv")

        print(file.exists())

        df = pd.read_csv(file)

        print(df.head())
        # df = pd.read_csv(file)
        # #df = pd.read_csv(file)
        # print(df.head(1))
            
        def Clean_text(text):
            if pd.isna(text):
                return "" 
            text=str(text)

            text=re.sub(r'\n',' ',text)
            text=re.sub(r'\t',' ',text)
            text=re.sub(r'\s+',' ',text)

            return text.strip()
        df['COMMENTS']=df['COMMENTS'].apply(Clean_text)
        df['PROCESS']=df['PROCESS'].apply(Clean_text)
        df['LOG_TEXT']=df['LOG_TEXT'].apply(Clean_text)

        df['Document']=("Issue : " + df['COMMENTS'] +
                        "ROOT Casue :"+df['PROCESS']+
                        "Pkg : "+df['LOG_TEXT']
                        )
        df.to_csv(self.save_file,index=False)

        return df


        
        




In [ ]:
try:
    config = ConfigurationManager().get_data_cleaning_config()

    preprocessing = DataCleaning(config)

    df = preprocessing.Data_Preprocessing()

    print(df.head())

except Exception as e:
    raise e

Config\Config.yaml
path config yaml Config\Config.yaml
Reading YAML : Config\Config.yaml
[2026-06-23 19:03:09,650: INFO: Common: YAML file loaded successfully : Config\Config.yaml]
Reading YAML : params.yaml
[2026-06-23 19:03:09,659: INFO: Common: YAML file loaded successfully : params.yaml]
CONFIG CONTENT:
{'artifacts_root': 'artifacts', 'Data_Ingestion': {'root_dir': 'artifacts/DataIngestion', 'oracledbconnection': {'host': 'localhost', 'port': 1521, 'service': 'free', 'username': 'SYSTEM', 'password': 'sys'}, 'source_table': 'COM_LOG', 'local_data_file': 'artifacts/DataIngestion/com_data_log.csv'}, 'Data_Cleaning': {'root_dir': 'artifacts/Data_Cleaning', 'source_path': 'artifacts/DataIngestion/com_data_log.csv', 'local_data_file': 'artifacts/DataIngestion/pre_processed_log.csv'}}
AVAILABLE KEYS:
['artifacts_root', 'Data_Ingestion', 'Data_Cleaning']
[2026-06-23 19:03:09,665: INFO: Common: Created directory : artifacts/Data_Cleaning]


Reading: artifacts\DataIngestion\com_data_log.csv
True
    ID                  TYPE                           PROCESS  SUB_PROCESS  \
0  111                   GSK                       DATAINESERT  INSERT_DATA   
1  112  EXTRACT_TRACKING_HUB                          JAZZ_STD          NaN   
2  113  EXTRACT_TRACKING_HUB                      ICONIS,HUBV5          NaN   
3  114  EXTRACT_TRACKING_HUB  PRA_EHB.TEVA_CUST.TEVA_CUST_MAIN          INS   
4  115  EXTRACT_TRACKING_HUB   PRA_EHB.TEVA_CUST.TEVA_CUST_OUT          NaN   

                    LOG_TEXT STATUS  \
0  ORA-01722: invalid number  ERROR   
1       JAZZHUBV512-MAR-2020  ERROR   
2    ICONIS,HUBV512-MAR-2020  ERROR   
3                 TEVA,HUBV5  ERROR   
4                        NaN  ERROR   

                                            COMMENTS  
0                          ORA-01722: invalid number  
1  |RAN PKG_HUB_ENGINE.CREATE_TABLE(7316328)|RAN ...  
2  CANNOT UPDATE RECORD FOR |*** ERROR WITH AWS T...  
3  |RAN PKG_HUB